# 02_block_cam_sensitivity_mel_nv

Qualitative block sensitivity check for MEL vs NV CAMs.

Goal:
- Compare where heatmaps focus across transformer blocks.
- Use this as an internal visual analysis notebook.
- Notebook 03 can then do quantitative block analysis.

Default blocks:
`[-1, -2, -4, -6, -8, -10, -12]`

Output design:
- Generate panels for CLS and GAP separately.
- Build one PDF per model.
- Each PDF page = one image.
- Each page has one row per block.
- Columns: RGB lesion outline, Grad CAM target, Diff CAM, Finer CAM.


In [1]:
from pathlib import Path
import json
import subprocess
import shlex
import pandas as pd

# =============================================================================
# 1. MAIN PARAMETERS
# =============================================================================

# Notebook location: notebooks/mel_nv/02_block_cam_sensitivity_mel_nv.ipynb
# REPO_ROOT = Path("../..").resolve() # local notebook
REPO_ROOT = Path("..").resolve() # ubelix notebook
REPO_ROOT = REPO_ROOT / "master-thesis"
HAM_ROOT = REPO_ROOT / "data" / "HAM10000"
MEL_NV_ROOT = HAM_ROOT / "mel_nv"

IMG_DIR = HAM_ROOT
MASK_ROOT = HAM_ROOT

# Keep qualitative first. all_test with 7 blocks x 2 models is heavy.
# Recommended: qualitative
SAMPLE_MODE = "all_test"  # "qualitative" or "all_test"

# Requested qualitative block sweep
TARGET_BLOCK_INDICES = [-1, -2, -4, -6, -8, -10, -12]
# TARGET_BLOCK_INDICES = [-1, -4, -10]

# Set True first if you only want to inspect commands.
DRY_RUN = False
RUN_PANEL_GENERATION = True
BUILD_PDFS = False  # important for all_test

# If None, use all rows in the selected CSV.
# For faster debugging, set e.g. NUM_SAMPLES_OVERRIDE = 2
NUM_SAMPLES_OVERRIDE = None
USE_BALANCED_CLASS_RECOMMENDATION = True

# For block sensitivity, keep columns compact.
# Full version possible: rgb_gt_mask,gradcam_a,gradcam_b,map_diff,finercam
PANEL_ITEMS = "rgb_gt_mask,gradcam_a,map_diff,finercam"
PANEL_SUFFIX = PANEL_ITEMS.replace(",", "_")

# Finer-CAM comparison strength
ALPHA = 0.8

# Quantitative CAM lesion alignment
TOP_HEAT_RATIO = 0.10
CAM_METRIC_METHOD = "finercam"  # "gradcam_target", "gradcam_diff", or "finercam"

# Paths
CLEAN_CSV = MEL_NV_ROOT / "ham_mel_nv_clean.csv"
QUAL_CSV = MEL_NV_ROOT / "ham_mel_nv_clean_qualitative_10_per_class_seed42.csv"

CLS_CKPT = REPO_ROOT / "external" / "checkpoints5" / "checkpoint-best-cls-ha5.pth"
GAP_CKPT = REPO_ROOT / "external" / "checkpoints5" / "checkpoint-best-gap-ha5.pth"

GT_COL = "gt_label"
CLASS_ARGS = ["--class_names", "MEL,NV"]
COMPARE_ARGS = [
    "--compare_mode", "gt_pair",
    "--A", "MEL",
    "--B", "NV",
    "--topk_compare", "1",
]

SCENARIOS = [
    {
        "name": "CLS HA 5.0",
        "short_name": "cls_ha5",
        "checkpoint": CLS_CKPT,
        "checkpoint_model_type": "panderm",
        "pooling": "cls",
    },
    {
        "name": "GAP HA 5.0",
        "short_name": "gap_ha5",
        "checkpoint": GAP_CKPT,
        "checkpoint_model_type": "panderm",
        "pooling": "mean",
    },
]

OUT_ROOT = REPO_ROOT / "outputs" / "mel_nv" / f"block_cam_sensitivity_{SAMPLE_MODE}"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("OUT_ROOT:", OUT_ROOT)
print("CLEAN_CSV exists:", CLEAN_CSV.exists(), CLEAN_CSV)
print("QUAL_CSV exists:", QUAL_CSV.exists(), QUAL_CSV)
print("CLS_CKPT exists:", CLS_CKPT.exists(), CLS_CKPT)
print("GAP_CKPT exists:", GAP_CKPT.exists(), GAP_CKPT)
print("Blocks:", TARGET_BLOCK_INDICES)

for scenario in SCENARIOS:
    print("\n", scenario["name"])
    print("  checkpoint:", scenario["checkpoint"])
    print("  pooling:", scenario["pooling"])
    if not Path(scenario["checkpoint"]).exists():
        print("  [WARN] missing checkpoint")


REPO_ROOT: /storage/homefs/cn21m021/projects/master-thesis
OUT_ROOT: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/block_cam_sensitivity_all_test
CLEAN_CSV exists: True /storage/homefs/cn21m021/projects/master-thesis/data/HAM10000/mel_nv/ham_mel_nv_clean.csv
QUAL_CSV exists: True /storage/homefs/cn21m021/projects/master-thesis/data/HAM10000/mel_nv/ham_mel_nv_clean_qualitative_10_per_class_seed42.csv
CLS_CKPT exists: True /storage/homefs/cn21m021/projects/master-thesis/external/checkpoints5/checkpoint-best-cls-ha5.pth
GAP_CKPT exists: True /storage/homefs/cn21m021/projects/master-thesis/external/checkpoints5/checkpoint-best-gap-ha5.pth
Blocks: [-1, -2, -4, -6, -8, -10, -12]

 CLS HA 5.0
  checkpoint: /storage/homefs/cn21m021/projects/master-thesis/external/checkpoints5/checkpoint-best-cls-ha5.pth
  pooling: cls

 GAP HA 5.0
  checkpoint: /storage/homefs/cn21m021/projects/master-thesis/external/checkpoints5/checkpoint-best-gap-ha5.pth
  pooling: mean


## 2. Build active CSV


In [2]:
def safe_name(name: str) -> str:
    out = str(name).lower()
    for ch in [" ", "/", "\\", ":", ";", ",", "(", ")", "[", "]", "{", "}", "+", "."]:
        out = out.replace(ch, "_")
    while "__" in out:
        out = out.replace("__", "_")
    return out.strip("_")


def make_active_csv() -> tuple[Path, int, pd.DataFrame]:
    csv_out_dir = OUT_ROOT / "csv"
    csv_out_dir.mkdir(parents=True, exist_ok=True)

    if SAMPLE_MODE == "qualitative":
        active_csv = QUAL_CSV
        df = pd.read_csv(active_csv)
    elif SAMPLE_MODE == "all_test":
        df = pd.read_csv(CLEAN_CSV)
        df = df[df["split"].astype(str).str.lower().eq("test")].copy()
        df = df.sort_values(["gt_label", "image_id"]).reset_index(drop=True)
        active_csv = csv_out_dir / "ham_mel_nv_clean_all_test.csv"
        df.to_csv(active_csv, index=False)
    else:
        raise ValueError("SAMPLE_MODE must be 'qualitative' or 'all_test'.")

    if NUM_SAMPLES_OVERRIDE is None:
        num_samples = len(df)
    else:
        num_samples = min(int(NUM_SAMPLES_OVERRIDE), len(df))

    print("ACTIVE_CSV:", active_csv)
    print("NUM_SAMPLES:", num_samples)
    display(df.head())
    display(df.head(num_samples).groupby(["split", "gt_label"]).size().unstack(fill_value=0))
    return active_csv, num_samples, df.head(num_samples).copy()


ACTIVE_CSV, NUM_SAMPLES, DISPLAY_DF = make_active_csv()


ACTIVE_CSV: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/block_cam_sensitivity_all_test/csv/ham_mel_nv_clean_all_test.csv
NUM_SAMPLES: 1021


,lesion_id,image_id,image,dx,gt_label,label,label_2class,binary_label,split,image_rel_path,...,cue_applied,cue_mask_rel_path,cue_mode,dx_type,age,sex,localization,dataset,age_group,dx_norm
0,HAM_0005846,ISIC_0024459,ISIC_0024459.jpg,mel,MEL,4,0,0,test,images/ISIC_0024459.jpg,...,False,NaN,clean,histo,80.0,male,back,vienna_dias,old,mel
1,HAM_0006699,ISIC_0024571,ISIC_0024571.jpg,mel,MEL,4,0,0,test,images/ISIC_0024571.jpg,...,False,NaN,clean,histo,65.0,male,face,rosendahl,old,mel
2,HAM_0000210,ISIC_0024624,ISIC_0024624.jpg,mel,MEL,4,0,0,test,images/ISIC_0024624.jpg,...,False,NaN,clean,histo,75.0,female,face,vidir_modern,old,mel
3,HAM_0005467,ISIC_0024640,ISIC_0024640.jpg,mel,MEL,4,0,0,test,images/ISIC_0024640.jpg,...,False,NaN,clean,histo,55.0,female,back,vienna_dias,old,mel
4,HAM_0007272,ISIC_0024756,ISIC_0024756.jpg,mel,MEL,4,0,0,test,images/ISIC_0024756.jpg,...,False,NaN,clean,histo,60.0,male,lower extremity,rosendahl,old,mel


gt_label,MEL,NV
split,,
test,70,951


## 3. Generate CAM panels for all blocks

This calls `scripts.generate_finer_cam_panderm` once per model and block.

Output folder pattern:

`outputs/mel_nv/block_cam_sensitivity_<mode>/panels/<model>/block_<block>/`


In [3]:
import torch
print(torch.cuda.is_available()) # True
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no cuda") # NVIDIA GeForce RTX 4090

True
NVIDIA GeForce RTX 4090


In [4]:
def run_command(cmd: list[str], dry_run: bool = False):
    print("\n" + "=" * 100)
    print(" ".join(shlex.quote(str(x)) for x in cmd))
    print("=" * 100)
    if dry_run:
        return
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)


def block_dir_name(block_index: int) -> str:
    return f"block_{block_index}".replace("-", "minus")


def generate_panels():
    panel_root = OUT_ROOT / "panels"
    panel_root.mkdir(parents=True, exist_ok=True)

    for scenario in SCENARIOS:
        for block_index in TARGET_BLOCK_INDICES:
            scenario_out_dir = panel_root / scenario["short_name"] / block_dir_name(block_index)
            scenario_out_dir.mkdir(parents=True, exist_ok=True)

            cmd = [
                "python", "-m", "scripts.generate_finer_cam_panderm",
                "--csv", str(ACTIVE_CSV),
                "--image_col", "image_rel_path",
                "--img_dir", str(IMG_DIR),
                "--gt_col", GT_COL,
                "--checkpoint", str(scenario["checkpoint"]),
                "--checkpoint_model_type", scenario.get("checkpoint_model_type", "panderm"),
                "--pooling", scenario["pooling"],
                "--out_dir", str(scenario_out_dir),
                "--num_samples", str(NUM_SAMPLES),
                "--method", "finercam",
                "--alpha", str(ALPHA),
                "--panel_items", PANEL_ITEMS,
                "--mask_root", str(MASK_ROOT),
                "--mask_col", "mask_rel_path",
                "--target_block_index", str(block_index),
                "--clinician_labels",
                "--model_display_name", f"{scenario['name']} block {block_index}",
                # "--save_json",
                "--save_raw_cams",
            ]
            cmd += CLASS_ARGS
            cmd += COMPARE_ARGS

            print(f"\nGenerating: {scenario['name']} | block {block_index}")
            run_command(cmd, dry_run=DRY_RUN)


if RUN_PANEL_GENERATION:
    generate_panels()
else:
    print("RUN_PANEL_GENERATION=False, skipping CAM/panel generation.")



Generating: CLS HA 5.0 | block -1

python -m scripts.generate_finer_cam_panderm --csv /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/block_cam_sensitivity_all_test/csv/ham_mel_nv_clean_all_test.csv --image_col image_rel_path --img_dir /storage/homefs/cn21m021/projects/master-thesis/data/HAM10000 --gt_col gt_label --checkpoint /storage/homefs/cn21m021/projects/master-thesis/external/checkpoints5/checkpoint-best-cls-ha5.pth --checkpoint_model_type panderm --pooling cls --out_dir /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/block_cam_sensitivity_all_test/panels/cls_ha5/block_minus1 --num_samples 1021 --method finercam --alpha 0.8 --panel_items rgb_gt_mask,gradcam_a,map_diff,finercam --mask_root /storage/homefs/cn21m021/projects/master-thesis/data/HAM10000 --mask_col mask_rel_path --target_block_index -1 --clinician_labels --model_display_name 'CLS HA 5.0 block -1' --save_raw_cams --class_names MEL,NV --compare_mode gt_pair --A MEL --B NV --topk_compare 1

/storage/homefs/cn21m021/projects/master-thesis/scripts/generate_finer_cam_panderm.py:716: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(checkpoint_path, m

[warn] load_state_dict mismatch: missing=2, unexpected=2
  missing sample: ['norm.weight', 'norm.bias']
  unexpected sample: ['fc_norm.weight', 'fc_norm.bias']
[info] Loaded PanDerm Base FT from checkpoint-best-cls-ha5.pth
[info] checkpoint_model_type= panderm pooling= cls use_seg_gate= False seg_gate_bg_keep= 0.15 seg_gate_detach= True
normalization method:  imagenet
[info] CAM target layer: blocks[11].norm1 (requested -1)
[debug compare_mode] gt_pair | A_idx=0 (MEL) | B_idx=1 (NV) | comparison=[1]
[saved raw cams] /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/block_cam_sensitivity_all_test/panels/cls_ha5/block_minus1/raw_cams/ISIC_0024459
[info] images/ISIC_0024459.jpg: A=0(MEL)  B=1(NV)  comparison=[NV]  top3=[MEL: 0.991, NV: 0.009]
[debug compare_mode] gt_pair | A_idx=0 (MEL) | B_idx=1 (NV) | comparison=[1]
[saved raw cams] /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/block_cam_sensitivity_all_test/panels/cls_ha5/block_minus1/raw_cams/ISIC_0024571

/storage/homefs/cn21m021/projects/master-thesis/scripts/generate_finer_cam_panderm.py:716: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(checkpoint_path, m

[warn] load_state_dict mismatch: missing=2, unexpected=2
  missing sample: ['norm.weight', 'norm.bias']
  unexpected sample: ['fc_norm.weight', 'fc_norm.bias']
[info] Loaded PanDerm Base FT from checkpoint-best-cls-ha5.pth
[info] checkpoint_model_type= panderm pooling= cls use_seg_gate= False seg_gate_bg_keep= 0.15 seg_gate_detach= True
normalization method:  imagenet
[info] CAM target layer: blocks[10].norm1 (requested -2)
[debug compare_mode] gt_pair | A_idx=0 (MEL) | B_idx=1 (NV) | comparison=[1]
[saved raw cams] /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/block_cam_sensitivity_all_test/panels/cls_ha5/block_minus2/raw_cams/ISIC_0024459
[info] images/ISIC_0024459.jpg: A=0(MEL)  B=1(NV)  comparison=[NV]  top3=[MEL: 0.991, NV: 0.009]
[debug compare_mode] gt_pair | A_idx=0 (MEL) | B_idx=1 (NV) | comparison=[1]
[saved raw cams] /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/block_cam_sensitivity_all_test/panels/cls_ha5/block_minus2/raw_cams/ISIC_0024571

...

[info] Loaded PanDerm Base FT from checkpoint-best-gap-ha5.pth
[info] checkpoint_model_type= panderm pooling= mean use_seg_gate= False seg_gate_bg_keep= 0.15 seg_gate_detach= True
normalization method:  imagenet
[info] CAM target layer: blocks[4].norm1 (requested -8)
[debug compare_mode] gt_pair | A_idx=0 (MEL) | B_idx=1 (NV) | comparison=[1]
[saved raw cams] /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/block_cam_sensitivity_all_test/panels/gap_ha5/block_minus8/raw_cams/ISIC_0024459
[info] images/ISIC_0024459.jpg: A=0(MEL)  B=1(NV)  comparison=[NV]  top3=[MEL: 0.894, NV: 0.106]
[debug compare_mode] gt_pair | A_idx=0 (MEL) | B_idx=1 (NV) | comparison=[1]
[saved raw cams] /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/block_cam_sensitivity_all_test/panels/gap_ha5/block_minus8/raw_cams/ISIC_0024571
[info] images/ISIC_0024571.jpg: A=0(MEL)  B=1(NV)  comparison=[NV]  top3=[MEL: 0.884, NV: 0.116]
[debug compare_mode] gt_pair | A_idx=0 (MEL) | B_idx=1 (NV) | 

/storage/homefs/cn21m021/projects/master-thesis/scripts/generate_finer_cam_panderm.py:716: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(checkpoint_path, m

[info] Loaded PanDerm Base FT from checkpoint-best-gap-ha5.pth
[info] checkpoint_model_type= panderm pooling= mean use_seg_gate= False seg_gate_bg_keep= 0.15 seg_gate_detach= True
normalization method:  imagenet
[info] CAM target layer: blocks[2].norm1 (requested -10)
[debug compare_mode] gt_pair | A_idx=0 (MEL) | B_idx=1 (NV) | comparison=[1]
[saved raw cams] /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/block_cam_sensitivity_all_test/panels/gap_ha5/block_minus10/raw_cams/ISIC_0024459
[info] images/ISIC_0024459.jpg: A=0(MEL)  B=1(NV)  comparison=[NV]  top3=[MEL: 0.894, NV: 0.106]
[debug compare_mode] gt_pair | A_idx=0 (MEL) | B_idx=1 (NV) | comparison=[1]
[saved raw cams] /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/block_cam_sensitivity_all_test/panels/gap_ha5/block_minus10/raw_cams/ISIC_0024571
[info] images/ISIC_0024571.jpg: A=0(MEL)  B=1(NV)  comparison=[NV]  top3=[MEL: 0.884, NV: 0.116]
[debug compare_mode] gt_pair | A_idx=0 (MEL) | B_idx=1 (NV)

/storage/homefs/cn21m021/projects/master-thesis/scripts/generate_finer_cam_panderm.py:716: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(checkpoint_path, m

[info] Loaded PanDerm Base FT from checkpoint-best-gap-ha5.pth
[info] checkpoint_model_type= panderm pooling= mean use_seg_gate= False seg_gate_bg_keep= 0.15 seg_gate_detach= True
normalization method:  imagenet
[info] CAM target layer: blocks[0].norm1 (requested -12)
[debug compare_mode] gt_pair | A_idx=0 (MEL) | B_idx=1 (NV) | comparison=[1]
[saved raw cams] /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/block_cam_sensitivity_all_test/panels/gap_ha5/block_minus12/raw_cams/ISIC_0024459
[info] images/ISIC_0024459.jpg: A=0(MEL)  B=1(NV)  comparison=[NV]  top3=[MEL: 0.894, NV: 0.106]
[debug compare_mode] gt_pair | A_idx=0 (MEL) | B_idx=1 (NV) | comparison=[1]
[saved raw cams] /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/block_cam_sensitivity_all_test/panels/gap_ha5/block_minus12/raw_cams/ISIC_0024571
[info] images/ISIC_0024571.jpg: A=0(MEL)  B=1(NV)  comparison=[NV]  top3=[MEL: 0.884, NV: 0.116]
[debug compare_mode] gt_pair | A_idx=0 (MEL) | B_idx=1 (NV)

## 4. Quantitative lesion alignment per block

This computes simple lesion alignment metrics from saved raw CAMs:

- `top10_inside`: fraction of top 10% hottest CAM pixels inside the lesion mask
- `pointing_game`: whether the hottest CAM pixel is inside the lesion mask
- `inside_mean`: mean CAM value inside lesion
- `outside_mean`: mean CAM value outside lesion

These metrics do not prove clinical correctness, but they help choose a model/block where the heatmap is at least lesion-oriented.

In [ ]:
import numpy as np
from PIL import Image
import cv2

def image_id_to_stem(image_id_value: str) -> str:
    p = Path(str(image_id_value))
    stem = p.stem if p.suffix else p.name
    return stem.replace("/", "_").replace("\\", "_").replace(" ", "_")


def resolve_from_root(root: Path, value) -> Path:
    p = Path(str(value))
    if p.is_absolute():
        return p
    return (root / p).resolve()


def minmax_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x).squeeze().astype(np.float32)
    x = x - np.nanmin(x)
    denom = np.nanmax(x) + 1e-8
    return x / denom


def load_mask_for_row(row: pd.Series, target_hw: tuple[int, int]) -> np.ndarray:
    if "mask_rel_path" not in row.index or pd.isna(row["mask_rel_path"]):
        raise ValueError("mask_rel_path missing")

    mask_path = resolve_from_root(MASK_ROOT, row["mask_rel_path"])
    if not mask_path.exists():
        raise FileNotFoundError(mask_path)

    mask = Image.open(mask_path).convert("L")
    mask = np.array(mask)
    mask = (mask > 0).astype(np.uint8)

    h, w = target_hw
    if mask.shape[:2] != (h, w):
        mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)

    return mask.astype(bool)


def image_id_to_stem_from_row(row: pd.Series) -> str:
    if "image_id" in row.index and pd.notna(row["image_id"]):
        return image_id_to_stem(row["image_id"])
    if "image_rel_path" in row.index and pd.notna(row["image_rel_path"]):
        return image_id_to_stem(row["image_rel_path"])
    if "image" in row.index and pd.notna(row["image"]):
        return image_id_to_stem(row["image"])
    raise ValueError("Could not infer image id/stem from row.")


def find_raw_cam_file(scenario: dict, block_index: int, row: pd.Series, method: str) -> Path | None:
    out_dir = OUT_ROOT / "panels" / scenario["short_name"] / block_dir_name(block_index)
    raw_dir = out_dir / "raw_cams"

    if not raw_dir.exists():
        return None

    stem_candidates = []
    if "image_id" in row.index and pd.notna(row["image_id"]):
        stem_candidates.append(image_id_to_stem(row["image_id"]))
    if "image_rel_path" in row.index and pd.notna(row["image_rel_path"]):
        stem_candidates.append(image_id_to_stem(row["image_rel_path"]))
    if "image" in row.index and pd.notna(row["image"]):
        stem_candidates.append(image_id_to_stem(row["image"]))

    # Common likely patterns
    for stem in dict.fromkeys(stem_candidates):
        patterns = [
            f"{stem}_{method}.npy",
            f"{stem}_cam_{method}.npy",
            f"{stem}*{method}*.npy",
        ]
        for pattern in patterns:
            matches = sorted(raw_dir.glob(pattern))
            if matches:
                return matches[0]

    # Fallback: inspect all npy files for this image stem
    for stem in dict.fromkeys(stem_candidates):
        matches = sorted(raw_dir.glob(f"{stem}*.npy"))
        method_matches = [p for p in matches if method in p.name]
        if method_matches:
            return method_matches[0]
        if matches and method == "finercam":
            # fallback only if names are not method-specific
            return matches[0]

    return None


def compute_alignment_metrics(cam: np.ndarray, mask: np.ndarray, top_ratio: float = 0.10) -> dict:
    cam = minmax_np(cam)

    if cam.shape[:2] != mask.shape[:2]:
        h, w = mask.shape[:2]
        cam = cv2.resize(cam, (w, h), interpolation=cv2.INTER_LINEAR)
        # cam = minmax_np(cam)

    mask_bool = mask.astype(bool)

    if mask_bool.sum() == 0:
        return {
            "top10_inside": np.nan,
            "pointing_game": np.nan,
            "inside_mean": np.nan,
            "outside_mean": np.nan,
        }

    flat_cam = cam.reshape(-1)
    flat_mask = mask_bool.reshape(-1)

    k = max(1, int(round(float(top_ratio) * flat_cam.size)))
    top_idx = np.argpartition(flat_cam, -k)[-k:]
    top_inside = float(flat_mask[top_idx].mean())

    max_idx = int(np.argmax(flat_cam))
    pointing_game = float(flat_mask[max_idx])

    inside_mean = float(cam[mask_bool].mean())
    outside_mean = float(cam[~mask_bool].mean()) if (~mask_bool).sum() > 0 else np.nan

    return {
        "top10_inside": top_inside,
        "pointing_game": pointing_game,
        "inside_mean": inside_mean,
        "outside_mean": outside_mean,
    }


def compute_block_alignment_table():
    rows = []

    for scenario in SCENARIOS:
        for block_index in TARGET_BLOCK_INDICES:
            for _, row in DISPLAY_DF.iterrows():
                image_id = str(row.get("image_id", image_id_to_stem_from_row(row)))
                gt_label = str(row.get(GT_COL, row.get("gt_label", "unknown")))

                cam_path = find_raw_cam_file(
                    scenario=scenario,
                    block_index=block_index,
                    row=row,
                    method=CAM_METRIC_METHOD,
                )

                if cam_path is None:
                    rows.append({
                        "model": scenario["name"],
                        "model_short": scenario["short_name"],
                        "gt_label": gt_label,
                        "image_id": image_id,
                        "block": block_index,
                        "cam_method": CAM_METRIC_METHOD,
                        "top10_inside": np.nan,
                        "pointing_game": np.nan,
                        "inside_mean": np.nan,
                        "outside_mean": np.nan,
                        "status": "missing_cam",
                    })
                    continue

                try:
                    cam = np.load(cam_path)
                    cam = minmax_np(cam)
                    mask = load_mask_for_row(row, target_hw=cam.shape[:2])
                    metrics = compute_alignment_metrics(cam, mask, top_ratio=TOP_HEAT_RATIO)

                    rows.append({
                        "model": scenario["name"],
                        "model_short": scenario["short_name"],
                        "gt_label": gt_label,
                        "image_id": image_id,
                        "block": block_index,
                        "cam_method": CAM_METRIC_METHOD,
                        **metrics,
                        "status": "ok",
                        "cam_path": str(cam_path),
                    })

                except Exception as e:
                    rows.append({
                        "model": scenario["name"],
                        "model_short": scenario["short_name"],
                        "gt_label": gt_label,
                        "image_id": image_id,
                        "block": block_index,
                        "cam_method": CAM_METRIC_METHOD,
                        "top10_inside": np.nan,
                        "pointing_game": np.nan,
                        "inside_mean": np.nan,
                        "outside_mean": np.nan,
                        "status": "failed",
                        "error": str(e),
                        "cam_path": str(cam_path),
                    })

    return pd.DataFrame(rows)


alignment_df = compute_block_alignment_table()

alignment_out = OUT_ROOT / f"block_cam_alignment_{CAM_METRIC_METHOD}_top{int(TOP_HEAT_RATIO * 100)}.csv"
alignment_df.to_csv(alignment_out, index=False)

print("Saved per-image alignment table:", alignment_out)
display(alignment_df[[
    "model",
    "gt_label",
    "image_id",
    "block",
    "top10_inside",
    "pointing_game",
    "inside_mean",
    "outside_mean",
    "status",
]].head(20))

print("Status counts:")
display(alignment_df["status"].value_counts(dropna=False))

Saved per-image alignment table: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/block_cam_sensitivity_all_test/block_cam_alignment_finercam_top10.csv


,model,gt_label,image_id,block,top10_inside,pointing_game,inside_mean,outside_mean,status
0,CLS HA 5.0,MEL,ISIC_0024459,-1,0.862894,1.0,0.669269,0.342076,ok
1,CLS HA 5.0,MEL,ISIC_0024571,-1,0.866481,1.0,0.445287,0.207766,ok
2,CLS HA 5.0,MEL,ISIC_0024624,-1,0.651056,1.0,0.822548,0.178037,ok
3,CLS HA 5.0,MEL,ISIC_0024640,-1,0.821244,1.0,0.499757,0.352385,ok
4,CLS HA 5.0,MEL,ISIC_0024756,-1,0.813870,1.0,0.774988,0.194696,ok
5,CLS HA 5.0,MEL,ISIC_0024886,-1,0.687724,1.0,0.673464,0.163086,ok
6,CLS HA 5.0,MEL,ISIC_0024967,-1,0.664209,1.0,0.711335,0.196812,ok
7,CLS HA 5.0,MEL,ISIC_0025105,-1,0.879833,1.0,0.522712,0.256972,ok
8,CLS HA 5.0,MEL,ISIC_0025132,-1,0.874053,1.0,0.791346,0.289036,ok
9,CLS HA 5.0,MEL,ISIC_0025234,-1,0.504384,0.0,0.655622,0.274534,ok


Status counts:


status
ok    14294
Name: count, dtype: int64

## 5. Build block sensitivity PDFs

This creates one PDF per model:
- `block_cam_sensitivity_cls_ha05_<mode>.pdf`
- `block_cam_sensitivity_gap_ha025_<mode>.pdf`

Each page is one image. Each row is one transformer block.

In [6]:
from PIL import Image, ImageDraw, ImageFont


def get_font(size: int, bold: bool = False):
    candidates = [
        "/System/Library/Fonts/Supplemental/Arial Bold.ttf" if bold else "/System/Library/Fonts/Supplemental/Arial.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf" if bold else "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    ]
    for path in candidates:
        if path and Path(path).exists():
            return ImageFont.truetype(path, size=size)
    return ImageFont.load_default()


FONT_TITLE = get_font(34, bold=True)
FONT_SUBTITLE = get_font(22, bold=False)
FONT_LABEL = get_font(22, bold=True)
FONT_SMALL = get_font(17, bold=False)


def find_panel_png(scenario: dict, block_index: int, row: pd.Series) -> Path | None:
    out_dir = OUT_ROOT / "panels" / scenario["short_name"] / block_dir_name(block_index)
    candidates = []
    if "image_rel_path" in row and pd.notna(row["image_rel_path"]):
        candidates.append(image_id_to_stem(row["image_rel_path"]))
    if "image_id" in row and pd.notna(row["image_id"]):
        candidates.append(image_id_to_stem(row["image_id"]))
    if "image" in row and pd.notna(row["image"]):
        candidates.append(image_id_to_stem(row["image"]))

    for stem in dict.fromkeys(candidates):
        direct = out_dir / f"{stem}_{PANEL_SUFFIX}.png"
        if direct.exists():
            return direct
        matches = sorted(out_dir.glob(f"{stem}_*.png"))
        if matches:
            return matches[0]
    return None


def make_page_for_image_and_model(row: pd.Series, scenario: dict) -> Image.Image:
    page_width = 2200
    margin = 45
    label_width = 210
    gap = 14
    title_h = 115
    available_panel_width = page_width - 2 * margin - label_width - gap

    loaded_panels = []
    for block_index in TARGET_BLOCK_INDICES:
        panel_path = find_panel_png(scenario, block_index, row)
        if panel_path is None:
            loaded_panels.append((block_index, None, None))
            continue

        panel = Image.open(panel_path).convert("RGB")
        scale = available_panel_width / panel.width
        new_h = int(panel.height * scale)
        panel = panel.resize((available_panel_width, new_h), Image.Resampling.LANCZOS)
        loaded_panels.append((block_index, panel, panel_path))

    row_heights = [panel.height if panel is not None else 190 for _, panel, _ in loaded_panels]
    page_height = title_h + margin + sum(row_heights) + gap * (len(row_heights) - 1) + margin
    page = Image.new("RGB", (page_width, page_height), "white")
    draw = ImageDraw.Draw(page)

    image_id = row.get("image_id", row.get("image_rel_path", "unknown"))
    gt = row.get(GT_COL, row.get("gt_label", "unknown"))
    title = f"Block CAM sensitivity: {scenario['name']}"
    subtitle = f"Image: {image_id} | Ground truth: {gt} | Pooling: {scenario['pooling']} | Mode: {SAMPLE_MODE}"
    draw.text((margin, 26), title, fill="black", font=FONT_TITLE)
    draw.text((margin, 75), subtitle, fill=(60, 60, 60), font=FONT_SUBTITLE)

    y = title_h
    for block_index, panel, panel_path in loaded_panels:
        row_h = panel.height if panel is not None else 190
        label_x = margin
        label_y = y + 20
        draw.text((label_x, label_y), f"Block {block_index}", fill="black", font=FONT_LABEL)
        draw.text((label_x, label_y + 32), "qualitative", fill=(80, 80, 80), font=FONT_SMALL)

        if panel is None:
            box_x = margin + label_width + gap
            box_y = y
            draw.rectangle([box_x, box_y, box_x + available_panel_width, box_y + row_h], outline=(180, 180, 180), width=2)
            draw.text((box_x + 30, box_y + 65), "Missing panel PNG", fill=(160, 0, 0), font=FONT_LABEL)
        else:
            page.paste(panel, (margin + label_width + gap, y))
        y += row_h + gap

    return page


def build_pdf_for_scenario(scenario: dict):
    pages = []
    for _, row in DISPLAY_DF.iterrows():
        pages.append(make_page_for_image_and_model(row, scenario))

    if not pages:
        raise RuntimeError("No pages generated.")

    pdf_out = OUT_ROOT / f"block_cam_sensitivity_{scenario['short_name']}_{SAMPLE_MODE}.pdf"
    pages[0].save(pdf_out, save_all=True, append_images=pages[1:], resolution=150.0)
    print("Saved PDF:", pdf_out)
    return pdf_out


PDF_OUTPUTS = []

if DRY_RUN:
    print("DRY_RUN=True, skipping PDF build.")
elif not BUILD_PDFS:
    print("BUILD_PDFS=False, skipping PDF build.")
else:
    for scenario in SCENARIOS:
        PDF_OUTPUTS.append(build_pdf_for_scenario(scenario))


BUILD_PDFS=False, skipping PDF build.


## 6. Save config and quick checks


In [7]:
config = {
    "sample_mode": SAMPLE_MODE,
    "target_block_indices": TARGET_BLOCK_INDICES,
    "num_samples": NUM_SAMPLES,
    "active_csv": str(ACTIVE_CSV),
    "out_root": str(OUT_ROOT),
    "panel_items": PANEL_ITEMS,
    "alpha": ALPHA,
    "pdf_outputs": [str(p) for p in PDF_OUTPUTS],
    "scenarios": [
        {
            "name": s["name"],
            "short_name": s["short_name"],
            "checkpoint": str(s["checkpoint"]),
            "checkpoint_model_type": s.get("checkpoint_model_type", "panderm"),
            "pooling": s["pooling"],
        }
        for s in SCENARIOS
    ],
    "class_args": CLASS_ARGS,
    "compare_args": COMPARE_ARGS,
}

config_out = OUT_ROOT / f"block_cam_sensitivity_config_{SAMPLE_MODE}.json"
config_out.write_text(json.dumps(config, indent=2))
print("Saved config:", config_out)

for scenario in SCENARIOS:
    print("\n" + "=" * 80)
    print(scenario["name"])
    for block_index in TARGET_BLOCK_INDICES:
        out_dir = OUT_ROOT / "panels" / scenario["short_name"] / block_dir_name(block_index)
        pngs = sorted(out_dir.glob("*.png"))
        metas = sorted(out_dir.glob("*_meta.json"))
        raw_dir = out_dir / "raw_cams"
        raw_files = sorted(raw_dir.glob("*.npy")) if raw_dir.exists() else []
        print(f"  block {block_index:>3}: png={len(pngs):>4} meta={len(metas):>4} raw_flat={len(raw_files):>4}")

print("\nPDF outputs:")
for p in PDF_OUTPUTS:
    print(" ", p)


Saved config: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/block_cam_sensitivity_all_test/block_cam_sensitivity_config_all_test.json

CLS HA 5.0
  block  -1: png=1021 meta=   0 raw_flat=4084
  block  -2: png=1021 meta=   0 raw_flat=4084
  block  -4: png=1021 meta=   0 raw_flat=4084
  block  -6: png=1021 meta=   0 raw_flat=4084
  block  -8: png=1021 meta=   0 raw_flat=4084
  block -10: png=1021 meta=   0 raw_flat=4084
  block -12: png=1021 meta=   0 raw_flat=4084

GAP HA 5.0
  block  -1: png=1021 meta=   0 raw_flat=4084
  block  -2: png=1021 meta=   0 raw_flat=4084
  block  -4: png=1021 meta=   0 raw_flat=4084
  block  -6: png=1021 meta=   0 raw_flat=4084
  block  -8: png=1021 meta=   0 raw_flat=4084
  block -10: png=1021 meta=   0 raw_flat=4084
  block -12: png=1021 meta=   0 raw_flat=4084

PDF outputs:


## 7. Practical interpretation checklist

When reviewing the PDFs, look for:

- Does the heatmap stay inside the lesion outline?
- Does it focus on lesion border, pigment network, dark/irregular areas, or clinically plausible structures?
- Does it focus on artifacts such as hair, dark corners, ruler marks, or background?
- Do later blocks become more concentrated or more diffuse?
- Does CLS behave differently from GAP?
- Which block gives stable maps across both MEL and NV examples?

Do not choose the final block only from this notebook. Use this to form hypotheses, then confirm quantitatively in notebook 03.

In [8]:
ok_alignment_df = alignment_df[alignment_df["status"].eq("ok")].copy()

if len(ok_alignment_df) == 0:
    raise RuntimeError("No valid CAM alignment rows found. Check raw CAM file naming/path.")

# =============================================================================
# 1. Add extra per-image diagnostic columns
# =============================================================================

ok_alignment_df["inside_outside_gap"] = (
    ok_alignment_df["inside_mean"] - ok_alignment_df["outside_mean"]
)

# This will be filled below if not already available.
# It estimates how much top10_inside we would expect from a random heatmap.
# For a random heatmap, expected top10_inside is roughly lesion_area_fraction.
if "mask_area_fraction" not in ok_alignment_df.columns:
    mask_area_fractions = []

    for _, row in ok_alignment_df.iterrows():
        try:
            # Need original row from DISPLAY_DF to load the mask.
            match = DISPLAY_DF[DISPLAY_DF["image_id"].astype(str).eq(str(row["image_id"]))]
            if len(match) == 0:
                mask_area_fractions.append(np.nan)
                continue

            original_row = match.iloc[0]
            cam = np.load(row["cam_path"])
            cam = minmax_np(cam)
            mask = load_mask_for_row(original_row, target_hw=cam.shape[:2])
            mask_area_fractions.append(float(mask.mean()))
        except Exception:
            mask_area_fractions.append(np.nan)

    ok_alignment_df["mask_area_fraction"] = mask_area_fractions

ok_alignment_df["top10_minus_random"] = (
    ok_alignment_df["top10_inside"] - ok_alignment_df["mask_area_fraction"]
)

# =============================================================================
# 2. Class-specific summary
# =============================================================================

alignment_summary = (
    ok_alignment_df
    .groupby(["model", "model_short", "gt_label", "block"], dropna=False)
    .agg(
        n=("image_id", "count"),
        top10_inside=("top10_inside", "mean"),
        pointing_game=("pointing_game", "mean"),
        inside_mean=("inside_mean", "mean"),
        outside_mean=("outside_mean", "mean"),
        inside_outside_gap=("inside_outside_gap", "mean"),
        mask_area_fraction=("mask_area_fraction", "mean"),
        top10_minus_random=("top10_minus_random", "mean"),
    )
    .reset_index()
)

# Class-specific score.
alignment_summary["selection_score"] = (
    0.45 * alignment_summary["top10_inside"]
    + 0.25 * alignment_summary["pointing_game"]
    + 0.20 * alignment_summary["inside_outside_gap"]
    + 0.10 * alignment_summary["top10_minus_random"]
)

summary_out = OUT_ROOT / f"block_cam_alignment_summary_by_class_{CAM_METRIC_METHOD}_top{int(TOP_HEAT_RATIO * 100)}.csv"
alignment_summary.to_csv(summary_out, index=False)

print("Saved summary by class:", summary_out)
display(alignment_summary.sort_values(["model", "gt_label", "selection_score"], ascending=[True, True, False]))

# =============================================================================
# 3. Original overall imbalanced summary
# =============================================================================

overall_summary = (
    ok_alignment_df
    .groupby(["model", "model_short", "block"], dropna=False)
    .agg(
        n=("image_id", "count"),
        top10_inside=("top10_inside", "mean"),
        pointing_game=("pointing_game", "mean"),
        inside_mean=("inside_mean", "mean"),
        outside_mean=("outside_mean", "mean"),
        inside_outside_gap=("inside_outside_gap", "mean"),
        mask_area_fraction=("mask_area_fraction", "mean"),
        top10_minus_random=("top10_minus_random", "mean"),
    )
    .reset_index()
)

overall_summary["selection_score"] = (
    0.45 * overall_summary["top10_inside"]
    + 0.25 * overall_summary["pointing_game"]
    + 0.20 * overall_summary["inside_outside_gap"]
    + 0.10 * overall_summary["top10_minus_random"]
)

overall_out = OUT_ROOT / f"block_cam_alignment_summary_overall_imbalanced_{CAM_METRIC_METHOD}_top{int(TOP_HEAT_RATIO * 100)}.csv"
overall_summary.to_csv(overall_out, index=False)

print("Saved imbalanced overall summary:", overall_out)
display(overall_summary.sort_values(["model", "selection_score"], ascending=[True, False]))

# =============================================================================
# 4. Balanced per-class summary
# =============================================================================
# Important:
# This gives MEL and NV equal weight, regardless of the number of images.
# This is the main summary you should use for block recommendation.

balanced_summary = (
    alignment_summary
    .groupby(["model", "model_short", "block"], dropna=False)
    .agg(
        n_classes=("gt_label", "nunique"),
        mean_n_per_class=("n", "mean"),
        balanced_top10_inside=("top10_inside", "mean"),
        balanced_pointing_game=("pointing_game", "mean"),
        balanced_inside_mean=("inside_mean", "mean"),
        balanced_outside_mean=("outside_mean", "mean"),
        balanced_inside_outside_gap=("inside_outside_gap", "mean"),
        balanced_mask_area_fraction=("mask_area_fraction", "mean"),
        balanced_top10_minus_random=("top10_minus_random", "mean"),
    )
    .reset_index()
)

balanced_summary["balanced_selection_score"] = (
    0.45 * balanced_summary["balanced_top10_inside"]
    + 0.25 * balanced_summary["balanced_pointing_game"]
    + 0.20 * balanced_summary["balanced_inside_outside_gap"]
    + 0.10 * balanced_summary["balanced_top10_minus_random"]
)

balanced_out = OUT_ROOT / f"block_cam_alignment_summary_balanced_by_class_{CAM_METRIC_METHOD}_top{int(TOP_HEAT_RATIO * 100)}.csv"
balanced_summary.to_csv(balanced_out, index=False)

print("Saved balanced per-class summary:", balanced_out)
display(balanced_summary.sort_values(["model", "balanced_selection_score"], ascending=[True, False]))

# =============================================================================
# 5. MEL-focused summary
# =============================================================================
# This is useful because MEL is clinically more important.
# It asks: which block is best specifically for melanoma examples?

mel_summary = alignment_summary[alignment_summary["gt_label"].astype(str).eq("MEL")].copy()

mel_summary = mel_summary.rename(columns={
    "top10_inside": "mel_top10_inside",
    "pointing_game": "mel_pointing_game",
    "inside_mean": "mel_inside_mean",
    "outside_mean": "mel_outside_mean",
    "inside_outside_gap": "mel_inside_outside_gap",
    "mask_area_fraction": "mel_mask_area_fraction",
    "top10_minus_random": "mel_top10_minus_random",
    "selection_score": "mel_selection_score",
})

mel_out = OUT_ROOT / f"block_cam_alignment_summary_mel_only_{CAM_METRIC_METHOD}_top{int(TOP_HEAT_RATIO * 100)}.csv"
mel_summary.to_csv(mel_out, index=False)

print("Saved MEL-focused summary:", mel_out)
display(mel_summary.sort_values(["model", "mel_selection_score"], ascending=[True, False]))

# =============================================================================
# 6. Join summaries for easier comparison
# =============================================================================

comparison_summary = balanced_summary.merge(
    overall_summary[[
        "model", "model_short", "block",
        "top10_inside", "pointing_game", "inside_outside_gap",
        "mask_area_fraction", "top10_minus_random", "selection_score"
    ]],
    on=["model", "model_short", "block"],
    how="left",
    suffixes=("", "_overall"),
)

comparison_summary = comparison_summary.merge(
    mel_summary[[
        "model", "model_short", "block",
        "mel_top10_inside", "mel_pointing_game",
        "mel_inside_outside_gap", "mel_top10_minus_random",
        "mel_selection_score"
    ]],
    on=["model", "model_short", "block"],
    how="left",
)

comparison_out = OUT_ROOT / f"block_cam_alignment_summary_comparison_{CAM_METRIC_METHOD}_top{int(TOP_HEAT_RATIO * 100)}.csv"
comparison_summary.to_csv(comparison_out, index=False)

print("Saved comparison summary:", comparison_out)
display(comparison_summary.sort_values(["model", "balanced_selection_score"], ascending=[True, False]))

# =============================================================================
# 7. Final recommendation print
# =============================================================================

print("\n" + "=" * 110)
print("FINAL BLOCK RECOMMENDATION WITH IMBALANCE CHECK")
print("=" * 110)
for model_name, sub_bal in balanced_summary.groupby("model"):
    sub_bal = sub_bal.sort_values(
        ["balanced_selection_score", "balanced_top10_inside", "balanced_pointing_game", "balanced_inside_outside_gap"],
        ascending=False,
    ).reset_index(drop=True)

    sub_overall = overall_summary[overall_summary["model"].eq(model_name)].sort_values(
        ["selection_score", "top10_inside", "pointing_game", "inside_outside_gap"],
        ascending=False,
    ).reset_index(drop=True)

    sub_mel = mel_summary[mel_summary["model"].eq(model_name)].sort_values(
        ["mel_selection_score", "mel_top10_inside", "mel_pointing_game", "mel_inside_outside_gap"],
        ascending=False,
    ).reset_index(drop=True)

    best_bal = sub_bal.iloc[0]
    best_overall = sub_overall.iloc[0]
    best_mel = sub_mel.iloc[0]

    print(f"\n{model_name}")
    print("-" * 90)

    if USE_BALANCED_CLASS_RECOMMENDATION:
        final_block = int(best_bal["block"])
        final_score = best_bal["balanced_selection_score"]
        final_reason = "balanced per-class score"
    else:
        final_block = int(best_overall["block"])
        final_score = best_overall["selection_score"]
        final_reason = "imbalanced overall score"

    print(
        f"FINAL chosen block: block {final_block} "
        f"based on {final_reason} "
        f"(score={final_score:.3f})."
    )

    print(
        f"Balanced per-class recommendation: block {int(best_bal['block'])} "
        f"(score={best_bal['balanced_selection_score']:.3f}, "
        f"top10_inside={best_bal['balanced_top10_inside']:.3f}, "
        f"pointing_game={best_bal['balanced_pointing_game']:.3f}, "
        f"inside_outside_gap={best_bal['balanced_inside_outside_gap']:.3f}, "
        f"top10_minus_random={best_bal['balanced_top10_minus_random']:.3f})."
    )

    print(
        f"MEL-only recommendation:           block {int(best_mel['block'])} "
        f"(score={best_mel['mel_selection_score']:.3f}, "
        f"top10_inside={best_mel['mel_top10_inside']:.3f}, "
        f"pointing_game={best_mel['mel_pointing_game']:.3f}, "
        f"inside_outside_gap={best_mel['mel_inside_outside_gap']:.3f}, "
        f"top10_minus_random={best_mel['mel_top10_minus_random']:.3f})."
    )

    print(
        f"Imbalanced overall recommendation: block {int(best_overall['block'])} "
        f"(score={best_overall['selection_score']:.3f}, "
        f"top10_inside={best_overall['top10_inside']:.3f}, "
        f"pointing_game={best_overall['pointing_game']:.3f}, "
        f"inside_outside_gap={best_overall['inside_outside_gap']:.3f}, "
        f"top10_minus_random={best_overall['top10_minus_random']:.3f})."
    )

    if int(best_bal["block"]) != int(best_overall["block"]):
        print(
            "[WARNING] Balanced recommendation differs from imbalanced overall recommendation. "
            "This means the original overall score was likely influenced by class imbalance."
        )

    if int(best_bal["block"]) != int(best_mel["block"]):
        print(
            "[NOTE] Balanced recommendation differs from MEL-only recommendation. "
            "For clinician-facing melanoma explanation, visually inspect the MEL-only best block carefully."
        )

print("\n" + "=" * 110)
print("Interpretation rule:")
print("- Use balanced per-class score as the main numeric block recommendation.")
print("- Use MEL-only score as safety-critical support because missed melanoma is clinically important.")
print("- Use imbalanced overall score only as a descriptive statistic, not as final decision logic.")
print("- Confirm final block visually in the qualitative PDF before showing clinician material.")
print("=" * 110)

Saved summary by class: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/block_cam_sensitivity_all_test/block_cam_alignment_summary_by_class_finercam_top10.csv


,model,model_short,gt_label,block,n,top10_inside,pointing_game,inside_mean,outside_mean,inside_outside_gap,mask_area_fraction,top10_minus_random,selection_score
6,CLS HA 5.0,cls_ha5,MEL,-1,70,0.801537,0.871429,0.672801,0.245993,0.426807,0.373812,0.427726,0.706683
4,CLS HA 5.0,cls_ha5,MEL,-4,70,0.604080,0.614286,0.244811,0.102146,0.142664,0.373812,0.230268,0.476967
3,CLS HA 5.0,cls_ha5,MEL,-6,70,0.449021,0.471429,0.161697,0.107795,0.053901,0.373812,0.075209,0.338218
1,CLS HA 5.0,cls_ha5,MEL,-10,70,0.344915,0.371429,0.077080,0.086358,-0.009278,0.373812,-0.028896,0.243324
2,CLS HA 5.0,cls_ha5,MEL,-8,70,0.253095,0.257143,0.090355,0.132168,-0.041812,0.373812,-0.120717,0.157744
5,CLS HA 5.0,cls_ha5,MEL,-2,70,0.263252,0.214286,0.239040,0.255440,-0.016401,0.373812,-0.110559,0.157699
0,CLS HA 5.0,cls_ha5,MEL,-12,70,0.219701,0.242857,0.271238,0.372717,-0.101479,0.373812,-0.154111,0.123873
7,CLS HA 5.0,cls_ha5,NV,-12,951,0.471787,0.500526,0.315384,0.146682,0.168703,0.261748,0.210039,0.392180
9,CLS HA 5.0,cls_ha5,NV,-8,951,0.424218,0.455310,0.292306,0.182379,0.109927,0.261748,0.162471,0.342958
8,CLS HA 5.0,cls_ha5,NV,-10,951,0.359536,0.356467,0.322879,0.226498,0.096381,0.261748,0.097788,0.279963


Saved imbalanced overall summary: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/block_cam_sensitivity_all_test/block_cam_alignment_summary_overall_imbalanced_finercam_top10.csv


,model,model_short,block,n,top10_inside,pointing_game,inside_mean,outside_mean,inside_outside_gap,mask_area_fraction,top10_minus_random,selection_score
0,CLS HA 5.0,cls_ha5,-12,1021,0.454504,0.482860,0.312358,0.162179,0.150179,0.269431,0.185073,0.373785
2,CLS HA 5.0,cls_ha5,-8,1021,0.412486,0.441724,0.278460,0.178937,0.099523,0.269431,0.143055,0.330260
1,CLS HA 5.0,cls_ha5,-10,1021,0.358534,0.357493,0.306027,0.216890,0.089137,0.269431,0.089103,0.277451
6,CLS HA 5.0,cls_ha5,-1,1021,0.342779,0.277179,0.386291,0.271587,0.114704,0.269431,0.073348,0.253821
3,CLS HA 5.0,cls_ha5,-6,1021,0.157627,0.138100,0.131249,0.236352,-0.105103,0.269431,-0.111804,0.073256
4,CLS HA 5.0,cls_ha5,-4,1021,0.110372,0.112635,0.072388,0.203712,-0.131324,0.269431,-0.159059,0.035655
5,CLS HA 5.0,cls_ha5,-2,1021,0.090113,0.067581,0.118754,0.298908,-0.180154,0.269431,-0.179318,0.003483
13,GAP HA 5.0,gap_ha5,-1,1021,0.728397,0.737512,0.651670,0.259086,0.392584,0.269431,0.458966,0.636570
8,GAP HA 5.0,gap_ha5,-10,1021,0.411382,0.421156,0.306132,0.183905,0.122227,0.269431,0.141951,0.329051
7,GAP HA 5.0,gap_ha5,-12,1021,0.242730,0.248776,0.219129,0.216470,0.002659,0.269431,-0.026701,0.169284


Saved balanced per-class summary: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/block_cam_sensitivity_all_test/block_cam_alignment_summary_balanced_by_class_finercam_top10.csv


...

Saved MEL-focused summary: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/block_cam_sensitivity_all_test/block_cam_alignment_summary_mel_only_finercam_top10.csv


,model,model_short,gt_label,block,n,mel_top10_inside,mel_pointing_game,mel_inside_mean,mel_outside_mean,mel_inside_outside_gap,mel_mask_area_fraction,mel_top10_minus_random,mel_selection_score
6,CLS HA 5.0,cls_ha5,MEL,-1,70,0.801537,0.871429,0.672801,0.245993,0.426807,0.373812,0.427726,0.706683
4,CLS HA 5.0,cls_ha5,MEL,-4,70,0.604080,0.614286,0.244811,0.102146,0.142664,0.373812,0.230268,0.476967
3,CLS HA 5.0,cls_ha5,MEL,-6,70,0.449021,0.471429,0.161697,0.107795,0.053901,0.373812,0.075209,0.338218
1,CLS HA 5.0,cls_ha5,MEL,-10,70,0.344915,0.371429,0.077080,0.086358,-0.009278,0.373812,-0.028896,0.243324
2,CLS HA 5.0,cls_ha5,MEL,-8,70,0.253095,0.257143,0.090355,0.132168,-0.041812,0.373812,-0.120717,0.157744
5,CLS HA 5.0,cls_ha5,MEL,-2,70,0.263252,0.214286,0.239040,0.255440,-0.016401,0.373812,-0.110559,0.157699
0,CLS HA 5.0,cls_ha5,MEL,-12,70,0.219701,0.242857,0.271238,0.372717,-0.101479,0.373812,-0.154111,0.123873
20,GAP HA 5.0,gap_ha5,MEL,-1,70,0.829109,0.871429,0.666132,0.279811,0.386322,0.373812,0.455298,0.713751
17,GAP HA 5.0,gap_ha5,MEL,-6,70,0.638462,0.657143,0.214896,0.079365,0.135531,0.373812,0.264650,0.505165
19,GAP HA 5.0,gap_ha5,MEL,-2,70,0.475497,0.471429,0.391140,0.298893,0.092247,0.373812,0.101685,0.360449


Saved comparison summary: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/block_cam_sensitivity_all_test/block_cam_alignment_summary_comparison_finercam_top10.csv


,model,model_short,block,n_classes,mean_n_per_class,balanced_top10_inside,balanced_pointing_game,balanced_inside_mean,balanced_outside_mean,balanced_inside_outside_gap,...,pointing_game,inside_outside_gap,mask_area_fraction,top10_minus_random,selection_score,mel_top10_inside,mel_pointing_game,mel_inside_outside_gap,mel_top10_minus_random,mel_selection_score
6,CLS HA 5.0,cls_ha5,-1,2,510.5,0.555274,0.552434,0.519001,0.259732,0.259269,...,0.277179,0.114704,0.269431,0.073348,0.253821,0.801537,0.871429,0.426807,0.427726,0.706683
1,CLS HA 5.0,cls_ha5,-10,2,510.5,0.352226,0.363948,0.199979,0.156428,0.043551,...,0.357493,0.089137,0.269431,0.089103,0.277451,0.344915,0.371429,-0.009278,-0.028896,0.243324
0,CLS HA 5.0,cls_ha5,-12,2,510.5,0.345744,0.371691,0.293311,0.259699,0.033612,...,0.482860,0.150179,0.269431,0.185073,0.373785,0.219701,0.242857,-0.101479,-0.154111,0.123873
2,CLS HA 5.0,cls_ha5,-8,2,510.5,0.338656,0.356227,0.191331,0.157273,0.034057,...,0.441724,0.099523,0.269431,0.143055,0.330260,0.253095,0.257143,-0.041812,-0.120717,0.157744
4,CLS HA 5.0,cls_ha5,-4,2,510.5,0.339056,0.344998,0.152254,0.156667,-0.004414,...,0.112635,-0.131324,0.269431,-0.159059,0.035655,0.604080,0.614286,0.142664,0.230268,0.476967
3,CLS HA 5.0,cls_ha5,-6,2,510.5,0.292600,0.292497,0.145352,0.176805,-0.031453,...,0.138100,-0.105103,0.269431,-0.111804,0.073256,0.449021,0.471429,0.053901,0.075209,0.338218
5,CLS HA 5.0,cls_ha5,-2,2,510.5,0.170311,0.135534,0.174470,0.278774,-0.104304,...,0.067581,-0.180154,0.269431,-0.179318,0.003483,0.263252,0.214286,-0.016401,-0.110559,0.157699
13,GAP HA 5.0,gap_ha5,-1,2,510.5,0.775047,0.799542,0.658369,0.268686,0.389683,...,0.737512,0.392584,0.269431,0.458966,0.636570,0.829109,0.871429,0.386322,0.455298,0.713751
8,GAP HA 5.0,gap_ha5,-10,2,510.5,0.385316,0.431208,0.207510,0.144664,0.062846,...,0.421156,0.122227,0.269431,0.141951,0.329051,0.355107,0.442857,-0.005971,-0.018704,0.267448
10,GAP HA 5.0,gap_ha5,-6,2,510.5,0.357481,0.362220,0.153393,0.183608,-0.030215,...,0.107738,-0.173234,0.269431,-0.154402,0.028610,0.638462,0.657143,0.135531,0.264650,0.505165



FINAL BLOCK RECOMMENDATION WITH IMBALANCE CHECK

CLS HA 5.0
------------------------------------------------------------------------------------------
FINAL chosen block: block -1 based on balanced per-class score (score=0.464).
Balanced per-class recommendation: block -1 (score=0.464, top10_inside=0.555, pointing_game=0.552, inside_outside_gap=0.259, top10_minus_random=0.237).
MEL-only recommendation:           block -1 (score=0.707, top10_inside=0.802, pointing_game=0.871, inside_outside_gap=0.427, top10_minus_random=0.428).
Imbalanced overall recommendation: block -12 (score=0.374, top10_inside=0.455, pointing_game=0.483, inside_outside_gap=0.150, top10_minus_random=0.185).
[WARNING] Balanced recommendation differs from imbalanced overall recommendation. This means the original overall score was likely influenced by class imbalance.

GAP HA 5.0
------------------------------------------------------------------------------------------
FINAL chosen block: block -1 based on balanced p

In [9]:
# =============================================================================
# Block ranking by class: easier interpretation table
# =============================================================================

ranked_by_class = alignment_summary.copy()

ranked_by_class["rank_within_model_class"] = (
    ranked_by_class
    .groupby(["model", "gt_label"])["selection_score"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

display(
    ranked_by_class
    .sort_values(["model", "gt_label", "rank_within_model_class"])
    [[
        "model",
        "gt_label",
        "block",
        "rank_within_model_class",
        "n",
        "top10_inside",
        "pointing_game",
        "inside_outside_gap",
        "mask_area_fraction",
        "top10_minus_random",
        "selection_score",
    ]]
)

ranked_by_class_out = OUT_ROOT / f"block_cam_alignment_ranked_by_class_{CAM_METRIC_METHOD}_top{int(TOP_HEAT_RATIO * 100)}.csv"
ranked_by_class.to_csv(ranked_by_class_out, index=False)
print("Saved:", ranked_by_class_out)

,model,gt_label,block,rank_within_model_class,n,top10_inside,pointing_game,inside_outside_gap,mask_area_fraction,top10_minus_random,selection_score
6,CLS HA 5.0,MEL,-1,1,70,0.801537,0.871429,0.426807,0.373812,0.427726,0.706683
4,CLS HA 5.0,MEL,-4,2,70,0.604080,0.614286,0.142664,0.373812,0.230268,0.476967
3,CLS HA 5.0,MEL,-6,3,70,0.449021,0.471429,0.053901,0.373812,0.075209,0.338218
1,CLS HA 5.0,MEL,-10,4,70,0.344915,0.371429,-0.009278,0.373812,-0.028896,0.243324
2,CLS HA 5.0,MEL,-8,5,70,0.253095,0.257143,-0.041812,0.373812,-0.120717,0.157744
5,CLS HA 5.0,MEL,-2,6,70,0.263252,0.214286,-0.016401,0.373812,-0.110559,0.157699
0,CLS HA 5.0,MEL,-12,7,70,0.219701,0.242857,-0.101479,0.373812,-0.154111,0.123873
7,CLS HA 5.0,NV,-12,1,951,0.471787,0.500526,0.168703,0.261748,0.210039,0.392180
9,CLS HA 5.0,NV,-8,2,951,0.424218,0.455310,0.109927,0.261748,0.162471,0.342958
8,CLS HA 5.0,NV,-10,3,951,0.359536,0.356467,0.096381,0.261748,0.097788,0.279963


Saved: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/block_cam_sensitivity_all_test/block_cam_alignment_ranked_by_class_finercam_top10.csv


## Flowchart Analysis: Old vs Current HA Implementation

### What Changed

Here is a precise diff across the three diagrams:

---

### Old HA (top-right pink flowchart)

| Step | Detail |
|---|---|
| Forward pass | `return_patch_tokens=True` only |
| Proxy map source | Patch tokens dot-producted with **classifier head weights** |
| Map type | ReLU proxy patch map (no true gradient signal) |
| HA loss | Computed directly from this proxy map vs mask |
| DAL | DAL projection maps computed from patch tokens |

**Core problem:** The proxy map was a simple linear projection of patch tokens through the classifier head. It carried no gradient information about which spatial locations actually influenced the prediction. It was a CAM approximation, not a GradCAM.

---

### Current HA (green flowchart / detailed bottom diagram)

| Step | Detail |
|---|---|
| Forward pass | `return_patch_tokens=True` + `store_attn=True` + `attn_layer=-1` |
| Attention stored | `last_attn` from final block |
| Gradient computation | `torch.autograd.grad(target_logits, last_attn, create_graph=True)` |
| Map source | **Attention weighted by its own gradient** (true Grad-weighted attention) |
| Pooling branch | GAP or CLS selects different attention rows and gradient rows |
| Map post-processing | ReLU, reshape 14x14, min-max normalize |
| HA loss | Dice/alignment loss on this **true GradCAM map** vs mask |
| Second-order gradients | `total_loss.backward()` flows through the `create_graph=True` GradCAM |

---

### Why GAP Block -1 Now Dominates Both Classes

This is the key insight from your block sensitivity results. Here is the causal chain:

**Old implementation:** The proxy map was `patch_tokens @ classifier_weights[gt_class]`. This has no notion of which attention head or which query-key interaction mattered. It treats all patch tokens symmetrically. For NV, the discriminative signal is spread more diffusely (homogeneous texture) while MEL has stronger focal signals (atypical structures). The proxy map could not capture this difference and NV needed early blocks where features were more spatially distributed.

**Current implementation:** The GradCAM map is `attn * grad(logit, attn)`. For GAP pooling specifically, you use `attn[:, :, 1:, 1:]` (patch-to-patch attention) and `grad[:, :, 1:, 1:]`. This means:
- Every patch token votes for how much attention between patch pairs influenced the target logit
- The gradient upweights attention patterns that were actually causal for the prediction
- For NV, the last block's patch-to-patch attention after full context aggregation is now a much richer signal than early blocks

For CLS pooling, you use `attn[:, :, 0, :]` (CLS-to-patch), which is a single row per head. The CLS token representation for NV may not have localized as cleanly, explaining why CLS NV still struggled at block -1.

**In short:** The switch from proxy map to true gradient-weighted attention is what fixed NV at block -1 for GAP. The gradient signal corrects for the diffuse NV feature distribution by only amplifying attention weights that actually changed the logit.

---

### Your Final Model Choice

GAP is the right primary choice for exactly the reason you stated. For **FinerCAM target vs reference class comparison**, you need the same block for both classes. GAP block -1 gives you:

- MEL top10 = 0.83
- NV top10 = 0.72

Both strong at the same block. CLS would force a compromise where NV CAMs at block -1 are unreliable (top10 = 0.31), which would make any cross-class comparison untrustworthy.

---

**One thing to note for your thesis narrative:** The improvement is not just architectural, it is conceptually important. Old HA trained on a **proxy** signal, which could be seen as circular (the map was derived directly from classifier weights with no gradient). Current HA trains on a signal that is **aligned with what the model actually used** to make the prediction. This is a stronger theoretical justification for human alignment training.

In [ ]:
# import matplotlib.pyplot as plt

# def overlay_top_heat_mask(row, cam, mask, top_ratio=0.10):
#     cam = minmax_np(cam)

#     if cam.shape[:2] != mask.shape[:2]:
#         h, w = mask.shape[:2]
#         cam = cv2.resize(cam, (w, h), interpolation=cv2.INTER_LINEAR)
#         cam = minmax_np(cam)

#     flat_cam = cam.reshape(-1)
#     k = max(1, int(round(top_ratio * flat_cam.size)))
#     top_idx = np.argpartition(flat_cam, -k)[-k:]

#     top_mask = np.zeros_like(flat_cam, dtype=bool)
#     top_mask[top_idx] = True
#     top_mask = top_mask.reshape(cam.shape[:2])

#     img_path = resolve_from_root(IMG_DIR, row["image_rel_path"])
#     img = Image.open(img_path).convert("RGB")
#     img = np.array(img)

#     h, w = mask.shape[:2]
#     if img.shape[:2] != (h, w):
#         img = cv2.resize(img, (w, h), interpolation=cv2.INTER_LINEAR)

#     return img, cam, mask, top_mask


# def visualize_top10_for_block(model_short, block_index, gt_label="MEL", n_examples=6):
#     sub = ok_alignment_df[
#         ok_alignment_df["model_short"].eq(model_short)
#         & ok_alignment_df["block"].eq(block_index)
#         & ok_alignment_df["gt_label"].astype(str).eq(gt_label)
#     ].copy()

#     sub = sub.sort_values("top10_inside", ascending=False).head(n_examples)

#     if len(sub) == 0:
#         print("No examples found.")
#         return

#     fig, axes = plt.subplots(len(sub), 4, figsize=(14, 3.5 * len(sub)))
#     if len(sub) == 1:
#         axes = np.expand_dims(axes, axis=0)

#     for ax_row, (_, metric_row) in zip(axes, sub.iterrows()):
#         original_row = DISPLAY_DF[DISPLAY_DF["image_id"].astype(str).eq(str(metric_row["image_id"]))].iloc[0]
#         cam = np.load(metric_row["cam_path"])
#         mask = load_mask_for_row(original_row, target_hw=minmax_np(cam).shape[:2])
#         img, cam, lesion_mask, top_mask = overlay_top_heat_mask(original_row, cam, mask, TOP_HEAT_RATIO)

#         ax_row[0].imshow(img)
#         ax_row[0].contour(lesion_mask, colors="red", linewidths=1)
#         ax_row[0].set_title(f"{metric_row['image_id']} | {gt_label}")

#         ax_row[1].imshow(cam, cmap="jet")
#         ax_row[1].set_title("CAM")

#         ax_row[2].imshow(img)
#         ax_row[2].imshow(top_mask, alpha=0.45, cmap="Reds")
#         ax_row[2].contour(lesion_mask, colors="blue", linewidths=1)
#         ax_row[2].set_title(f"Top {int(TOP_HEAT_RATIO*100)}% heat")

#         ax_row[3].imshow(img)
#         ax_row[3].imshow(lesion_mask, alpha=0.25, cmap="Blues")
#         ax_row[3].imshow(top_mask, alpha=0.45, cmap="Reds")
#         ax_row[3].set_title(
#             f"top_inside={metric_row['top10_inside']:.3f}\n"
#             f"random≈{metric_row['mask_area_fraction']:.3f}"
#         )

#         for ax in ax_row:
#             ax.axis("off")

#     plt.tight_layout()
#     out_path = OUT_ROOT / f"debug_top10_overlay_{model_short}_block_{block_index}_{gt_label}.png"
#     plt.savefig(out_path, dpi=200, bbox_inches="tight")
#     plt.show()
#     print("Saved:", out_path)


# # Examples:
# visualize_top10_for_block("cls_ha05", -10, gt_label="MEL", n_examples=6)
# visualize_top10_for_block("gap_ha025", -10, gt_label="MEL", n_examples=6)
# visualize_top10_for_block("cls_ha05", -1, gt_label="MEL", n_examples=6)
# visualize_top10_for_block("gap_ha025", -1, gt_label="MEL", n_examples=6)

No examples found.
No examples found.
No examples found.
No examples found.
